In [1]:
import pandas as pd
import re

# 1. Temizlenmiş veriyi yükle
v3 = pd.read_csv("../data/processed/arac_temiz_v3.csv")
bil = v3[v3["yakit_tipi"] == "Bilinmiyor"]

def yakit_adayi(brand, model, variant):
    """(yakıt, güven) döndürür. güven='kesin': açık işaret, güven='tahmin': eleme mantığı."""
    v = str(variant).upper()
    b = str(brand).lower()
    
    # Genel kural
    if re.search(r"\bMHEV\b", v): return "Hibrit/Elektrik", "kesin"
    
    # Marka bazlı kurallar
    if b == "bmw":
        if re.search(r"\d{2,3}D\b", v): return "Dizel", "kesin"
        if re.search(r"\d{2,3}I\b", v): return "Benzin", "kesin"
        if re.search(r"\d{2,3}E\b", v): return "Hibrit/Elektrik", "kesin"
        
    elif b == "mercedes":
        if re.search(r"\bD\b|\bCDI\b|\bBLUETEC\b", v): return "Dizel", "kesin"
        if re.search(r"\bCGI\b", v): return "Benzin", "kesin"
        # Eğer dizel işaretleri yoksa, muhtemelen benzindir (Tahmin kuralı)
        if not re.search(r"\bD\b|\bCDI\b|\bBLUETEC\b|\bCGI\b", v): return "Benzin", "tahmin"
        
    elif b == "opel":
        if re.search(r"\bCDTI\b|\bDTI\b", v): return "Dizel", "kesin"
        if re.search(r"\bECOTEC\b", v): return "Benzin", "tahmin"
        
    elif b == "honda":
        if "ECO" in v: return "LPG", "kesin"

    return None, None

# 2. Kuralları Bilinmiyor'lar üzerinde test et
sonuc = [yakit_adayi(b, m, v) for b, m, v in zip(bil["brand"], bil["model"], bil["variant"])]
bil = bil.assign(aday=[s[0] for s in sonuc], guven=[s[1] for s in sonuc])
yeni = bil[bil["aday"].notna()]

# 3. Sonuçları yazdır
print(f"{len(yeni)} ilan yeni sınıflandı | kalan Bilinmiyor: {bil['aday'].isna().sum()}")
print("\n--- Marka ve Aday Yakıt Dağılımı ---")
print(yeni.groupby(["brand", "aday"]).size())

# 4. Kör doğrulama için 60 satırlık örneklem al ve kaydet
dogrulama = yeni[["brand", "model", "variant"]].sample(60, random_state=21).copy()
dogrulama["gercek_yakit"] = ""
dogrulama.to_csv("../reports/yakit_dogrulama_2.csv", index=True)
print("\n>>> Kör doğrulama dosyası oluşturuldu: reports/yakit_dogrulama_2.csv")

29557 ilan yeni sınıflandı | kalan Bilinmiyor: 66476

--- Marka ve Aday Yakıt Dağılımı ---
brand     aday           
bmw       Benzin             12778
          Dizel               4401
          Hibrit/Elektrik       27
mercedes  Benzin              7115
          Dizel               5120
opel      Dizel                116
dtype: int64

>>> Kör doğrulama dosyası oluşturuldu: reports/yakit_dogrulama_2.csv
